In [1]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

StatementMeta(, 4bfbe23b-0aaa-4ea8-a06c-91344eca30d8, 3, Finished, Available, Finished, False)

In [2]:
df_silver = spark.sql("""
SELECT *
FROM bronze_sales
""")

StatementMeta(, 4bfbe23b-0aaa-4ea8-a06c-91344eca30d8, 4, Finished, Available, Finished, False)

In [3]:
display(df_silver)

StatementMeta(, 4bfbe23b-0aaa-4ea8-a06c-91344eca30d8, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0ce4be50-cd88-47ec-a67a-2ca2624f9e79)

In [4]:
df_silver.printSchema()

StatementMeta(, 4bfbe23b-0aaa-4ea8-a06c-91344eca30d8, 6, Finished, Available, Finished, False)

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



In [6]:
df_silver = df_silver.dropDuplicates()

StatementMeta(, 4bfbe23b-0aaa-4ea8-a06c-91344eca30d8, 8, Finished, Available, Finished, False)

In [8]:
df_silver.filter(
    col("CustomerID").isNull()
).count()

StatementMeta(, 4bfbe23b-0aaa-4ea8-a06c-91344eca30d8, 10, Finished, Available, Finished, False)

228826

In [9]:
df_silver = df_silver.filter(
    col("CustomerID").isNotNull()
)

StatementMeta(, 4bfbe23b-0aaa-4ea8-a06c-91344eca30d8, 11, Finished, Available, Finished, False)

In [10]:
df_silver = df_silver.filter(
    col("Description").isNotNull()
)

StatementMeta(, 4bfbe23b-0aaa-4ea8-a06c-91344eca30d8, 12, Finished, Available, Finished, False)

In [11]:
df_silver = df_silver.withColumn(
    "Description",
    trim(col("Description"))
)

StatementMeta(, 4bfbe23b-0aaa-4ea8-a06c-91344eca30d8, 13, Finished, Available, Finished, False)

In [12]:
df_silver = df_silver.withColumn(
    "InvoiceDate",
    to_timestamp(
        col("InvoiceDate"),
        "dd-MM-yyyy HH:mm"
    )
)

StatementMeta(, 4bfbe23b-0aaa-4ea8-a06c-91344eca30d8, 14, Finished, Available, Finished, False)

In [13]:
display(df_silver.select("InvoiceDate"))

StatementMeta(, 4bfbe23b-0aaa-4ea8-a06c-91344eca30d8, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9f08a2a3-8b91-4c3b-92f5-31161f66f60d)

In [14]:
df_silver = df_silver.filter(
    col("Quantity") > 0
)

StatementMeta(, 4bfbe23b-0aaa-4ea8-a06c-91344eca30d8, 16, Finished, Available, Finished, False)

In [15]:
df_silver = df_silver.withColumn(
    "Revenue",
    col("Quantity") * col("UnitPrice")
)

StatementMeta(, 4bfbe23b-0aaa-4ea8-a06c-91344eca30d8, 17, Finished, Available, Finished, False)

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `UnitPrice` cannot be resolved. Did you mean one of the following? [`InvoiceDate`, `Description`, `spark_catalog`.`chimcobldhq2agrledq6urb5e8g56obcclpi0gbec5m7it39cdpiakr1dhin6gbec5m7it39cdpluj31ddimgrrlediiap32ds`.`bronze_sales`.`Price`, `spark_catalog`.`chimcobldhq2agrledq6urb5e8g56obcclpi0gbec5m7it39cdpiakr1dhin6gbec5m7it39cdpluj31ddimgrrlediiap32ds`.`bronze_sales`.`Invoice`, `spark_catalog`.`chimcobldhq2agrledq6urb5e8g56obcclpi0gbec5m7it39cdpiakr1dhin6gbec5m7it39cdpluj31ddimgrrlediiap32ds`.`bronze_sales`.`Country`].;
'Project [Invoice#756, StockCode#757, Description#1551, Quantity#759, InvoiceDate#1561, Price#761, CustomerID#762, Country#763, ingestion_timestamp#764, (Quantity#759 * 'UnitPrice) AS Revenue#1655]
+- Filter (Quantity#759 > 0)
   +- Project [Invoice#756, StockCode#757, Description#1551, Quantity#759, to_timestamp(InvoiceDate#760, Some(dd-MM-yyyy HH:mm), TimestampType, Some(U

In [16]:
df_silver = df_silver.withColumn(
    "Revenue",
    col("Quantity") * col("Price")
)

StatementMeta(, 4bfbe23b-0aaa-4ea8-a06c-91344eca30d8, 18, Finished, Available, Finished, False)

In [19]:
display(
    df_silver.select(
        [
            count(when(col(c).isNull(), c)).alias(c)
            for c in df_silver.columns
        ]
    )
)

StatementMeta(, 4bfbe23b-0aaa-4ea8-a06c-91344eca30d8, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d9d01589-5694-44ad-8520-fa2c526ae53d)

In [20]:
df_silver.filter(
    col("Revenue") < 0
).count()

StatementMeta(, 4bfbe23b-0aaa-4ea8-a06c-91344eca30d8, 22, Finished, Available, Finished, False)

0

In [21]:
df_silver.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable("silver_sales")

StatementMeta(, 4bfbe23b-0aaa-4ea8-a06c-91344eca30d8, 23, Finished, Available, Finished, False)

In [22]:
spark.sql("""
SELECT *
FROM silver_sales
LIMIT 10
""").show()

StatementMeta(, 4bfbe23b-0aaa-4ea8-a06c-91344eca30d8, 24, Finished, Available, Finished, False)

+-------+---------+--------------------+--------+-------------------+-----+----------+--------------+--------------------+------------------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|CustomerID|       Country| ingestion_timestamp|           Revenue|
+-------+---------+--------------------+--------+-------------------+-----+----------+--------------+--------------------+------------------+
| 502588|    82580| BATHROOM METAL SIGN|       4|2010-03-25 12:43:00| 0.55|     16550|United Kingdom|2026-07-20 12:27:...|               2.2|
| 502599|    22383|LUNCH BAG SUKI  D...|       1|2010-03-25 13:16:00| 1.65|     17301|United Kingdom|2026-07-20 12:27:...|              1.65|
| 502605|    22457|NATURAL SLATE HEA...|      12|2010-03-25 13:43:00| 2.95|     14685|United Kingdom|2026-07-20 12:27:...|35.400000000000006|
| 502605|    21531|RETRO SPOT SUGAR ...|      24|2010-03-25 13:43:00|  2.1|     14685|United Kingdom|2026-07-20 12:27:...|50.400000000000006|
| 5026